# Milestone M3 - Eksperimen Utama (3 Seed), 4 Ablasi Anggota, Error Gallery & Grad-CAM (Topik T3-A)
**Kelompok 6 - Deep Learning A (Sistem Informasi, Universitas Trunojoyo Madura)**

Notebook ini dijalankan bersih dari nol dan langsung menyimpan seluruh log CSV, grafik, bobot model, serta arsip zip ke folder Google Drive (`101PbkgjCD-y_OaZPzRcqZzdgjoCf2ld8`):
1. **Perbandingan Utama Multi-Seed (`3 Seed: 42, 123, 999` x `2 Paradigma` x `2 Arah` x `4 Strategi Warna` = 48 Run):** `U-Net` vs `DeepLabV3+` pada `D1 -> D2` dan `D2 -> D1` (`mean +- std` + uji statistik berpasangan).
2. **4 Studi Ablasi Satu-Faktor (`seed=42`, 11 Run):** Attala (`Eval-Only` vs `Train+Eval`), Nafaul (`CE` vs `Dice` vs `CE+Dice`), M. Rafly (`128` vs `256`, `none` vs `geom`), Naufal (`ResNet-34 Full` vs `MobileNetV2` vs `ResNet-34 Frozen`).
3. **Analisis Kesalahan Kuantitatif (`NoAdapt` vs `Adaptasi Terbaik` selain NoAdapt pada kedua arah), Breakdown Ukuran Objek (`Kecil/Sedang/Besar`), & Grad-CAM (`Nukleus` dan `Sitoplasma`)** + **Tabel Efisiensi (`K7`)**.

In [ ]:
import os
import sys
import subprocess
import pandas as pd

REPO_URL = "https://github.com/attaramadhani/UAS-DL-Kelompok6-T3A-WBC-DomainShift.git"
REPO_NAME = "UAS-DL-Kelompok6-T3A-WBC-DomainShift"
GDRIVE_FOLDER_ID = "101PbkgjCD-y_OaZPzRcqZzdgjoCf2ld8"
RUN_FROM_SCRATCH = True  # True: mulai bersih dari nol; ubah ke False hanya jika ingin resume dari Google Drive

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != REPO_NAME:
        if not os.path.exists(REPO_NAME):
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)
    subprocess.run(["git", "pull", "--ff-only"], check=False)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2000)
pd.set_option("display.max_colwidth", 120)

import torch
from src.gdrive_sync import GDriveFolderSync

print(
    "PyTorch:",
    torch.__version__,
    "| CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
)

gdrive = GDriveFolderSync(folder_id=GDRIVE_FOLDER_ID, milestone_subdir="M3_Main_Experiments_XAI")
gdrive.connect(require_in_colab=True)

## 1. Persiapan Dataset, Split Tetap (`seed=42`) & Referensi Normalisasi (Train-Only)

In [ ]:
import csv
import gc
import os
import shutil
import time
import zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
from torch.utils.data import DataLoader

from src.dataset import (
    check_leakage_across_splits,
    download_and_extract_zheng,
    generate_fixed_splits,
    make_torch_dataset,
)
from src.stain_norm import MacenkoNormalizer, ReinhardNormalizer, select_representative_reference
from src.models import build_segmentation_model
from src.trainer import evaluate_loader, run_experiment
from src.xai_and_errors import (
    generate_gradcam_grid,
    get_gradcam_target_layer,
    records_to_dataframe,
    save_baseline_preview_grid,
    save_error_gallery,
    save_learning_curve_plot,
    summarize_by_object_size,
    summarize_error_categories,
)
from src.m3_reporting import (
    build_ablation_tables,
    build_efficiency_table,
    build_main_summary,
    build_statistical_tests,
    save_ablation_overview_figure,
    save_domain_shift_bar_figure,
)

domain_dirs = download_and_extract_zheng("data")
split_csvs = {"D1": "splits/zheng_d1_split.csv", "D2": "splits/zheng_d2_split.csv"}
if not all(os.path.exists(p) for p in split_csvs.values()):
    split_csvs = generate_fixed_splits(domain_dirs, "splits", seed=42)
leak = check_leakage_across_splits(split_csvs, max_dhash_dist=4)
print("Split tetap:", split_csvs)
print("Cek kebocoran -> exact:", leak["cross_split_leakage_exact"], "| dHash<=4:", leak["cross_split_leakage_dhash_le_4bits"])

def get_train_paths(dkey):
    with open(split_csvs[dkey], encoding="utf-8") as f:
        return [os.path.join(domain_dirs[dkey], r["image_file"]) for r in csv.DictReader(f) if r["split"] == "train"]

ref_train = {dkey: select_representative_reference(get_train_paths(dkey)) for dkey in ("D1", "D2")}
norm_macenko = {dkey: MacenkoNormalizer().fit(ref_train[dkey]) for dkey in ("D1", "D2")}
norm_reinhard = {dkey: ReinhardNormalizer().fit(ref_train[dkey]) for dkey in ("D1", "D2")}

LOG_PATH = "logs/m3_experiment_runs.csv"
HISTORY_DIR = "logs/histories"
WEIGHTS_DIR = "weights"

completed_runs = set()
if RUN_FROM_SCRATCH:
    if os.path.exists(LOG_PATH):
        os.remove(LOG_PATH)
    for folder in (HISTORY_DIR, WEIGHTS_DIR):
        if os.path.isdir(folder):
            shutil.rmtree(folder)
    os.makedirs("logs", exist_ok=True)
    os.makedirs(HISTORY_DIR, exist_ok=True)
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    print("Mode RUN_FROM_SCRATCH aktif: log & checkpoint lama dibersihkan agar mulai dari nol.")
else:
    if gdrive.connected:
        gdrive.restore_subdir("logs", "logs")
        gdrive.restore_subdir("weights", "weights")
    if os.path.exists(LOG_PATH):
        prev_df = pd.read_csv(LOG_PATH)
        prev_ok = prev_df[prev_df["status"] == "success"]
        prev_ok.to_csv(LOG_PATH, index=False)
        completed_runs = set(prev_ok["run_name"].tolist())
        print(f"Resume aktif: {len(completed_runs)} run sukses sebelumnya dipertahankan.")

if gdrive.connected:
    gdrive.sync_paths(["splits", "configs"])

print("Siap menjalankan seluruh eksperimen M3!")

## 2. Perbandingan Utama (`2 Paradigma` x `4 Strategi Warna` x `2 Arah` x `3 Seed` = `48 Run`)

In [ ]:
SEEDS = [42, 123, 999]
EPOCHS = 40
t_suite_start = time.perf_counter()

COLOR_STRATEGIES = [
    ("NoAdapt", "geom", None),
    ("ColorJitter", "color_geom", None),
    ("Macenko", "geom", "macenko"),
    ("Reinhard", "geom", "reinhard"),
]

main_plan = []
for train_dom, target_dom in [("D1", "D2"), ("D2", "D1")]:
    for arch in ["unet", "deeplabv3plus"]:
        for strat_name, aug_mode, norm_kind in COLOR_STRATEGIES:
            for seed in SEEDS:
                main_plan.append((train_dom, target_dom, arch, strat_name, aug_mode, norm_kind, seed))

for idx, (train_dom, target_dom, arch, strat_name, aug_mode, norm_kind, seed) in enumerate(main_plan, 1):
    name = f"Main_{arch}_{strat_name}_{train_dom}_to_{target_dom}_s{seed}"
    if name in completed_runs:
        print(f"[{idx}/{len(main_plan)}] SKIP (sudah selesai): {name}")
        continue

    norm_obj = None
    if norm_kind == "macenko":
        norm_obj = norm_macenko[train_dom]
    elif norm_kind == "reinhard":
        norm_obj = norm_reinhard[train_dom]

    ckpt_path = f"{WEIGHTS_DIR}/{name}.pth" if seed == 42 else None
    res = run_experiment(
        run_name=name,
        owner="Tim & Attala (230441100144)",
        strategy=strat_name,
        domain_dirs=domain_dirs,
        split_csvs=split_csvs,
        train_domain=train_dom,
        target_domain=target_dom,
        arch=arch,
        encoder_name="resnet34",
        loss_name="ce_dice",
        aug_mode=aug_mode,
        train_normalizer=norm_obj,
        eval_normalizer=norm_obj,
        seed=seed,
        epochs=EPOCHS,
        log_csv_path=LOG_PATH,
        save_weights_path=ckpt_path,
        capture_test_records=False,
        history_dir=HISTORY_DIR,
    )
    row = res["row"]
    elapsed_min = (time.perf_counter() - t_suite_start) / 60.0
    print(
        f"[{idx}/{len(main_plan)}] {name} | val_dice={row['val_dice_fg']:.4f} | "
        f"in_miou={row['in_miou_fg']:.4f} | cross_miou={row['cross_miou_fg']:.4f} "
        f"({row['duration_sec']:.1f}s, total {elapsed_min:.1f}m)"
    )

    if gdrive.connected:
        sync_list = [LOG_PATH, f"{HISTORY_DIR}/{name}.csv"]
        if ckpt_path and os.path.exists(ckpt_path):
            sync_list.append(ckpt_path)
        gdrive.sync_paths(sync_list)

    del res
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_runs = pd.read_csv(LOG_PATH)
flat_main, fmt_main = build_main_summary(df_runs)
stat_df = build_statistical_tests(df_runs)

flat_main.to_csv("logs/m3_main_summary.csv", index=False)
stat_df.to_csv("logs/m3_statistical_tests.csv", index=False)

print("\n=== TABEL RINGKASAN UTAMA 3-SEED (LENGKAP TANPA TERPOTONG) ===")
display(fmt_main)

print("\n=== UJI STATISTIK BERPASANGAN (VS NOADAPT & DEEPLABV3+ VS U-NET) ===")
display(stat_df)

if gdrive.connected:
    gdrive.sync_paths(["logs/m3_main_summary.csv", "logs/m3_statistical_tests.csv"])
    print("[GDrive] Tabel ringkasan utama & uji statistik tersimpan ke Google Drive.")

## 3. Studi Ablasi Satu-Faktor Masing-Masing Anggota (`11 Run`, `seed=42`) & Tabel Efisiensi (`K7`)

In [ ]:
ablation_queue = [
    dict(
        run_name="Abl_Attala_Macenko_EvalOnly",
        owner="Attala (230441100144)",
        strategy="Macenko_EvalOnly",
        train_domain="D1", target_domain="D2", arch="unet", encoder_name="resnet34",
        freeze_encoder=False, loss_name="ce_dice", aug_mode="geom",
        train_normalizer=None, eval_normalizer=norm_macenko["D1"], input_size=256,
    ),
    dict(
        run_name="Abl_Attala_Reinhard_EvalOnly",
        owner="Attala (230441100144)",
        strategy="Reinhard_EvalOnly",
        train_domain="D1", target_domain="D2", arch="unet", encoder_name="resnet34",
        freeze_encoder=False, loss_name="ce_dice", aug_mode="geom",
        train_normalizer=None, eval_normalizer=norm_reinhard["D1"], input_size=256,
    ),
    *[
        dict(
            run_name=f"Abl_Nafaul_Loss_{lname}",
            owner="Nafaul (240441100125)",
            strategy=f"Loss_{lname}",
            train_domain="D1", target_domain="D2", arch="unet", encoder_name="resnet34",
            freeze_encoder=False, loss_name=lname, aug_mode="geom",
            train_normalizer=None, eval_normalizer=None, input_size=256,
        )
        for lname in ["ce", "dice", "ce_dice"]
    ],
    *[
        dict(
            run_name=f"Abl_Rafly_Sz{sz}_{amode}",
            owner="M. Rafly (240441100086)",
            strategy=f"Sz{sz}_{amode}",
            train_domain="D1", target_domain="D2", arch="deeplabv3plus", encoder_name="resnet34",
            freeze_encoder=False, loss_name="ce_dice", aug_mode=amode,
            train_normalizer=None, eval_normalizer=None, input_size=sz,
        )
        for sz, amode in [(128, "geom"), (256, "none"), (256, "geom")]
    ],
    *[
        dict(
            run_name=f"Abl_Naufal_{enc}_{'Frozen' if frz else 'Full'}",
            owner="Naufal (240441100038)",
            strategy=f"{enc}_{'Frozen' if frz else 'Full'}",
            train_domain="D1", target_domain="D2", arch="deeplabv3plus", encoder_name=enc,
            freeze_encoder=frz, loss_name="ce_dice", aug_mode="geom",
            train_normalizer=None, eval_normalizer=None, input_size=256,
        )
        for enc, frz in [("resnet34", False), ("mobilenet_v2", False), ("resnet34", True)]
    ],
]

for idx, cfg in enumerate(ablation_queue, 1):
    rname = cfg["run_name"]
    if rname in completed_runs:
        print(f"[{idx}/{len(ablation_queue)}] SKIP (sudah selesai): {rname}")
        continue

    ckpt_path = f"{WEIGHTS_DIR}/{rname}.pth"
    res = run_experiment(
        domain_dirs=domain_dirs,
        split_csvs=split_csvs,
        seed=42,
        epochs=EPOCHS,
        log_csv_path=LOG_PATH,
        save_weights_path=ckpt_path,
        capture_test_records=False,
        history_dir=HISTORY_DIR,
        **cfg,
    )
    row = res["row"]
    print(
        f"[{idx}/{len(ablation_queue)}] {rname} | val_dice={row['val_dice_fg']:.4f} | "
        f"in_miou={row['in_miou_fg']:.4f} | cross_miou={row['cross_miou_fg']:.4f} ({row['duration_sec']:.1f}s)"
    )

    if gdrive.connected:
        gdrive.sync_paths([LOG_PATH, f"{HISTORY_DIR}/{rname}.csv", ckpt_path])

    del res
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_all = pd.read_csv(LOG_PATH)
print(f"\nTotal waktu GPU seluruh 59 eksperimen: {df_all['duration_sec'].sum() / 60.0:.2f} menit")

abl_combined, abl_dict = build_ablation_tables(df_all)
eff_df = build_efficiency_table(df_all)

abl_combined.to_csv("logs/m3_ablation_summary.csv", index=False)
eff_df.to_csv("logs/m3_efficiency_summary.csv", index=False)

show_cols = [
    "run_name", "strategy", "in_miou_fg", "in_dice_nuc", "in_dice_cyto",
    "cross_miou_fg", "cross_dice_nuc", "cross_dice_cyto", "cross_hd95_nuc", "cross_hd95_cyto", "delta_cross_miou_vs_ref",
]
for mkey, tbl in abl_dict.items():
    if tbl.empty:
        continue
    print(f"\n=== ABLASI {mkey.upper()}: {tbl['ablation_topic'].iloc[0]} ===")
    display(tbl[[c for c in show_cols if c in tbl.columns]])

print("\n=== TABEL EFISIENSI MODEL (K7: BATCH=1 SETELAH 10 WARM-UP, 50 CITRA) ===")
display(eff_df)

if gdrive.connected:
    gdrive.sync_paths(["logs/m3_ablation_summary.csv", "logs/m3_efficiency_summary.csv"])
    print("[GDrive] Tabel ablasi & efisiensi tersimpan ke Google Drive.")

## 4. Analisis Kesalahan Kuantitatif (`K6`), Per-Ukuran Objek (`K5`) & Grad-CAM (`Nukleus` dan `Sitoplasma`)
Menggunakan checkpoint beku `seed=42` dan memilih **strategi adaptasi terbaik non-baseline (`strategy != 'NoAdapt'`)** berdasarkan rerata 3 seed, sehingga galeri kesalahan tidak pernah bertabrakan dengan baseline.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
df_all = pd.read_csv(LOG_PATH)
main_ok = df_all[df_all["run_name"].str.startswith("Main_") & (df_all["status"] == "success")].copy()

def evaluate_saved_run(run_row, dkey):
    t_dom = run_row["train_domain"]
    norm_map = {
        "Macenko": norm_macenko[t_dom],
        "Reinhard": norm_reinhard[t_dom],
        "Macenko_EvalOnly": norm_macenko[t_dom],
        "Reinhard_EvalOnly": norm_reinhard[t_dom],
    }
    ev_norm = norm_map.get(run_row["strategy"], None)
    test_ds = make_torch_dataset(
        domain_dirs[dkey], split_csvs[dkey], "test",
        input_size=int(run_row["input_size"]), normalizer=ev_norm,
    )
    loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
    model = build_segmentation_model(
        arch=run_row["arch"],
        encoder_name=run_row["encoder"],
        encoder_weights=None,
        classes=3,
        freeze_encoder=bool(run_row["freeze_encoder"]),
    ).to(device)
    ckpt = f"{WEIGHTS_DIR}/{run_row['run_name']}.pth"
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    summary, records = evaluate_loader(model, loader, device=device, capture_records=True, compute_hd95=True)
    for r in records:
        r["domain"] = f"{t_dom}->{dkey} ({run_row['strategy']})"
    return model, summary, records

# --- Arah 1: D1 -> D2 (U-Net seed=42) ---
base_d1d2 = main_ok[main_ok["run_name"] == "Main_unet_NoAdapt_D1_to_D2_s42"].iloc[0]
best_strat_d1d2 = (
    main_ok[(main_ok["arch"] == "unet") & (main_ok["train_domain"] == "D1") & (main_ok["target_domain"] == "D2") & (main_ok["strategy"] != "NoAdapt")]
    .groupby("strategy")["cross_miou_fg"]
    .mean()
    .idxmax()
)
best_d1d2 = main_ok[main_ok["run_name"] == f"Main_unet_{best_strat_d1d2}_D1_to_D2_s42"].iloc[0]
print(f"[D1->D2] Baseline: {base_d1d2['run_name']} vs Adaptasi Terbaik (rerata 3-seed): {best_d1d2['run_name']}")

base_model, _, base_records = evaluate_saved_run(base_d1d2, "D2")
best_model, _, best_records = evaluate_saved_run(best_d1d2, "D2")

# --- Arah 2: D2 -> D1 (U-Net seed=42) untuk verifikasi kontras kegagalan vs pemulihan ---
base_d2d1 = main_ok[main_ok["run_name"] == "Main_unet_NoAdapt_D2_to_D1_s42"].iloc[0]
best_strat_d2d1 = (
    main_ok[(main_ok["arch"] == "unet") & (main_ok["train_domain"] == "D2") & (main_ok["target_domain"] == "D1") & (main_ok["strategy"] != "NoAdapt")]
    .groupby("strategy")["cross_miou_fg"]
    .mean()
    .idxmax()
)
best_d2d1 = main_ok[main_ok["run_name"] == f"Main_unet_{best_strat_d2d1}_D2_to_D1_s42"].iloc[0]
print(f"[D2->D1] Baseline: {base_d2d1['run_name']} vs Adaptasi Terbaik (rerata 3-seed): {best_d2d1['run_name']}")

base_d2_model, _, base_d2_records = evaluate_saved_run(base_d2d1, "D1")
best_d2_model, _, best_d2_records = evaluate_saved_run(best_d2d1, "D1")

# --- Ringkasan Frekuensi Kategori Kesalahan & Breakdown Ukuran Objek (K5 & K6) ---
err_rows = []
for dir_lbl, mod_lbl, recs in [
    ("D1->D2", "NoAdapt (Baseline)", base_records),
    ("D1->D2", f"{best_strat_d1d2} (Terbaik)", best_records),
    ("D2->D1", "NoAdapt (Baseline)", base_d2_records),
    ("D2->D1", f"{best_strat_d2d1} (Terbaik)", best_d2_records),
]:
    for cat, cnt in summarize_error_categories(recs).items():
        err_rows.append({"direction": dir_lbl, "model": mod_lbl, "error_category": cat, "count": cnt, "total_test": len(recs)})
df_err_summary = pd.DataFrame(err_rows)
df_err_summary.to_csv("logs/m3_error_analysis_summary.csv", index=False)
print("\n=== FREKUENSI KATEGORI KESALAHAN KUANTITATIF (K6) ===")
display(df_err_summary)

size_tables = pd.concat([
    summarize_by_object_size(base_records, strategy="NoAdapt", domain_label="D1->D2"),
    summarize_by_object_size(best_records, strategy=best_strat_d1d2, domain_label="D1->D2"),
    summarize_by_object_size(base_d2_records, strategy="NoAdapt", domain_label="D2->D1"),
    summarize_by_object_size(best_d2_records, strategy=best_strat_d2d1, domain_label="D2->D1"),
], ignore_index=True)
size_tables.to_csv("logs/m3_size_stratified_metrics.csv", index=False)
print("\n=== PERFORMA BERDASARKAN UKURAN OBJEK FOREGROUND (K5) ===")
display(size_tables)

df_xai_per_img = pd.concat([
    records_to_dataframe(base_records, strategy="NoAdapt", domain_label="D1->D2"),
    records_to_dataframe(best_records, strategy=best_strat_d1d2, domain_label="D1->D2"),
    records_to_dataframe(base_d2_records, strategy="NoAdapt", domain_label="D2->D1"),
    records_to_dataframe(best_d2_records, strategy=best_strat_d2d1, domain_label="D2->D1"),
], ignore_index=True)
df_xai_per_img.to_csv("logs/m3_xai_per_image_metrics.csv", index=False)

# --- Galeri Kesalahan & Peta Grad-CAM ---
os.makedirs("docs/figures", exist_ok=True)
save_error_gallery(base_records, "docs/figures/fig4a_error_gallery_baseline.png", top_k=8)
save_error_gallery(best_records, "docs/figures/fig4b_error_gallery_best_adapted.png", top_k=8)
save_baseline_preview_grid(
    base_d2_records,
    best_d2_records,
    out_path="docs/figures/fig4c_error_gallery_d2_to_d1_comparison.png",
    n_each=4,
    first_label="D2->D1 NoAdapt",
    second_label=f"D2->D1 {best_strat_d2d1}",
)

sorted_rec = sorted(best_records, key=lambda r: r["dice_fg_mean"])
worst_4 = [{**r, "tag": "Salah/Terburuk"} for r in sorted_rec[:4]]
best_4 = [{**r, "tag": "Benar/Terbaik"} for r in sorted_rec[-4:]]
target_layer = get_gradcam_target_layer(best_model)

generate_gradcam_grid(best_model, target_layer, best_4 + worst_4, "docs/figures/fig5a_gradcam_nucleus.png", target_class=2, device=device)
generate_gradcam_grid(best_model, target_layer, best_4 + worst_4, "docs/figures/fig5b_gradcam_cytoplasm.png", target_class=1, device=device)

for fname in [
    "fig4a_error_gallery_baseline.png",
    "fig4b_error_gallery_best_adapted.png",
    "fig4c_error_gallery_d2_to_d1_comparison.png",
    "fig5a_gradcam_nucleus.png",
    "fig5b_gradcam_cytoplasm.png",
]:
    print("===", fname, "===")
    display(Image(filename=f"docs/figures/{fname}"))

if gdrive.connected:
    gdrive.sync_paths([
        "logs/m3_error_analysis_summary.csv",
        "logs/m3_size_stratified_metrics.csv",
        "logs/m3_xai_per_image_metrics.csv",
        "docs/figures",
    ])
    print("[GDrive] Seluruh gambar XAI/Error Gallery & tabel analisis error tersimpan ke Google Drive.")

for m in (base_model, best_model, base_d2_model, best_d2_model):
    m.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Grafik Ringkasan Domain Shift (`3-Seed`), Kurva Pelatihan, Ringkasan Ablasi & Simpan Total ke Google Drive

In [ ]:
df_all = pd.read_csv(LOG_PATH)
_, abl_dict = build_ablation_tables(df_all)

save_domain_shift_bar_figure(df_all, "docs/figures/fig6_domain_shift_comparison.png")
save_learning_curve_plot(
    {
        "NoAdapt (U-Net D1->D2)": f"{HISTORY_DIR}/Main_unet_NoAdapt_D1_to_D2_s42.csv",
        "ColorJitter (U-Net D1->D2)": f"{HISTORY_DIR}/Main_unet_ColorJitter_D1_to_D2_s42.csv",
        "Macenko (U-Net D1->D2)": f"{HISTORY_DIR}/Main_unet_Macenko_D1_to_D2_s42.csv",
        "Reinhard (U-Net D1->D2)": f"{HISTORY_DIR}/Main_unet_Reinhard_D1_to_D2_s42.csv",
    },
    out_path="docs/figures/fig7_learning_curves.png",
    title="Kurva Pelatihan & Validasi Lintas Strategi (U-Net ResNet-34 D1->D2, seed=42)",
)
save_ablation_overview_figure(abl_dict, "docs/figures/fig8_ablation_summary.png")

for fig_name in [
    "fig6_domain_shift_comparison.png",
    "fig7_learning_curves.png",
    "fig8_ablation_summary.png",
]:
    print("===", fig_name, "===")
    display(Image(filename=f"docs/figures/{fig_name}"))

bundle_path = "m3_results_bundle.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("logs", "splits", "docs/figures", "configs"):
        if not os.path.exists(folder):
            continue
        for root, _, files in os.walk(folder):
            for fname in sorted(files):
                zf.write(os.path.join(root, fname))

print("Selesai! Arsip ringkas M3 dibuat di:", os.path.abspath(bundle_path))
if gdrive.connected:
    uploaded = gdrive.sync_paths(["logs", "splits", "docs/figures", "weights", "configs", bundle_path])
    print(f"[OK] {uploaded} berkas baru/diperbarui disinkronkan ke Google Drive: https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}")
    remote_df = pd.DataFrame(gdrive.list_remote_files())
    if not remote_df.empty:
        print(f"Total ukuran di '{gdrive.milestone_subdir}': {remote_df['size_mb'].sum():.2f} MB ({len(remote_df)} berkas)")
        display(remote_df[["drive_path", "size_mb", "file_id"]])